In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, MinMaxScaler, StandardScaler
from category_encoders import CountEncoder
from sklearn.compose import ColumnTransformer
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, Lasso
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,
    mean_absolute_error, mean_squared_error,
)

In [ ]:
from plot_utils import cm_plot, regression_plots

In [ ]:
current_dir = os.getcwd()

In [ ]:
flights_sampled_parquet = "flights_sampled.parquet"

try:
    df_flights_sampled = pd.read_parquet(os.path.join(current_dir, flights_sampled_parquet))
except FileNotFoundError:
    print(f"Arquivo {flights_sampled_parquet} não encontrado. Certifique-se de rodar o notebook 'us_flights_ml_eda_sampled.ipynb'.")
    raise

In [ ]:
df_flights_sampled.info()

In [ ]:
df_flights_sampled.dropna(inplace=True)

## Encoding / Scaling

In [ ]:
# Colunas categóricas não ordenadas de baixa cardinalidade (One-Hot Encoding)
ohe_cat_cols = ["AIRLINE"]

# Colunas categóricas não ordenadas de alta cardinalidade (Frequency Encoding)
freq_cat_cols = [
    "ORIGIN_AIRPORT", "ORIGIN_CITY", "ORIGIN_STATE",
    "DESTINATION_AIRPORT", "DESTINATION_CITY", "DESTINATION_STATE",
    "ROUTE",
]

# Colunas categóricas ordenadas (Ordinal Encoding + MinMax Scaler)
oe_cat_cols = ["DISTANCE_CATEGORY", "SCHEDULED_TIME_CATEGORY"]

dist_cat_labels = [
    "Short Distance",
    "Medium Distance",
    "Long Distance",
]

sch_time_cat_labels = [
    "Short Duration",
    "Medium Duration",
    "Long Duration",
    "Very Long Duration",
]

# Colunas categóricas ordenadas (MinMax Scaler)
sc_cat_cols = ["MONTH", "DAY", "DAY_OF_WEEK", "HOUR", "FLIGHT_SEQUENCE"]

# Colunas numéricas (MinMax Scaler)
minmax_num_cols = ["DISTANCE", "SCHEDULED_TIME", "DEPARTURE_ACC_MINUTES", "ARRIVAL_ACC_MINUTES"]

# Colunas numéricas (Standard Scaler)
std_num_cols = ["ORIGIN_LATITUDE", "ORIGIN_LONGITUDE", "DESTINATION_LATITUDE", "DESTINATION_LONGITUDE"]

# Colunas booleanas (Passthrough)
bool_cols = ["HOLIDAY_ORIGIN", "HOLIDAY_EVE_ORIGIN", "HOLIDAY_DESTINATION", "HOLIDAY_EVE_DESTINATION"]

preprocessor_standard = ColumnTransformer(
    transformers=[
        ("ohe", OneHotEncoder(handle_unknown="ignore", drop="first", sparse_output=False), ohe_cat_cols),
        ("freq", CountEncoder(handle_unknown="value", normalize=True), freq_cat_cols),
        ("ord", Pipeline([
            ("enc", OrdinalEncoder(handle_unknown="error", categories=[dist_cat_labels, sch_time_cat_labels])),
            ("scaler", MinMaxScaler()),
        ]), oe_cat_cols),
        ("ord_minmax_scaler", MinMaxScaler(), sc_cat_cols),
        ("minmax_scaler", MinMaxScaler(), minmax_num_cols),
        ("std_scaler", StandardScaler(), std_num_cols),
    ],
    remainder="passthrough"
)

preprocessor_dt = ColumnTransformer(
    transformers=[
        ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ohe_cat_cols),
        ("freq", CountEncoder(handle_unknown="value", normalize=True), freq_cat_cols),
        ("ord", OrdinalEncoder(handle_unknown="error", categories=[dist_cat_labels, sch_time_cat_labels]), oe_cat_cols),
    ],
    remainder="passthrough"
)

## Classification

In [ ]:
X = df_flights_sampled.drop(columns=["ARRIVAL_DELAY"])

tol_min = 15
y = df_flights_sampled["ARRIVAL_DELAY"] >= tol_min

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [ ]:
pipelines = {
    "LogisticRegression": Pipeline([
        ("preprocessor", clone(preprocessor_standard)),
        ("classifier", LogisticRegression(solver="liblinear", random_state=42, max_iter=1000)),
    ]),
    "SVM": Pipeline([
        ("preprocessor", clone(preprocessor_standard)),
        ("classifier", LinearSVC(random_state=42, max_iter=2000)),
    ]),
    "DecisionTree": Pipeline([
        ("preprocessor", clone(preprocessor_dt)),
        ("classifier", DecisionTreeClassifier(random_state=42)),
    ]),
}

In [ ]:
param_grids = {
    "LogisticRegression": {
        "classifier__C": [0.1, 1.0, 10.0],
        "classifier__l1_ratio": [1, 0],  # l1_ratio = 1 para penalty = "l1", l1_ratio = 0 para penalty = "l2"
        "classifier__class_weight": [None, "balanced"],
    },
    "SVM": {
        "classifier__C": [0.1, 1.0, 10.0],
        "classifier__class_weight": [None, "balanced"],
    },
    "DecisionTree": {
        "classifier__max_depth": [None, 5, 10],
        "classifier__min_samples_split": [2, 5, 20],
        "classifier__class_weight": [None, "balanced"],
    },
}

In [ ]:
for model_name, pipeline in pipelines.items():
    print(f"GridSearch - {model_name}")
    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grids[model_name],
        cv=5,
        scoring="f1",
        n_jobs=-1
    )
    grid_search.fit(X_train, y_train)

    print("CV Results:")
    cv_results_cols = [
        "params",
        "split0_test_score", "split1_test_score", "split2_test_score", "split3_test_score", "split4_test_score",
        "mean_test_score", "std_test_score",
    ]
    df_cv_results = pd.DataFrame(grid_search.cv_results_)[cv_results_cols]
    df_cv_results = pd.concat([
        pd.json_normalize(df_cv_results["params"]),
        df_cv_results.drop(columns=["params"]),
    ], axis=1)
    print(df_cv_results)
    print("\n")

    print(f"Best Parameters: {grid_search.best_params_}")
    print("\n")

    best_model = grid_search.best_estimator_
    y_pred = best_model.predict(X_test)

    print("Confusion Matrix:")
    cm = confusion_matrix(y_test, y_pred)
    cm_plot(cm, model_name)

    tn, fp, fn, tp = cm.ravel()
    print(f"TN: {tn}")
    print(f"FP: {fp}")
    print(f"FN: {fn}")
    print(f"TP: {tp}")
    print("\n")

    print("Metrics:")
    print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
    print(f"Precision: {precision_score(y_test, y_pred):.4f}")
    print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
    print(f"F1-Score:  {f1_score(y_test, y_pred):.4f}")
    print("\n")

    print("Feature Importance:")
    feature_names = best_model.named_steps["preprocessor"].get_feature_names_out()

    classifier = best_model.named_steps["classifier"]
    if hasattr(classifier, "feature_importances_"):
        feature_values = classifier.feature_importances_
    elif hasattr(classifier, "coef_"):
        feature_values = np.abs(classifier.coef_).flatten()
    else:
        print(f"Model {model_name} does not support feature importance")
        continue

    df_feature_importance = pd.DataFrame({
        "feature": feature_names,
        "importance": feature_values,
    }).sort_values("importance", ascending=False)
    print(df_feature_importance)
    print("\n")

## Regression

In [ ]:
X_reg = df_flights_sampled.drop(columns=["ARRIVAL_DELAY"])
y_reg = df_flights_sampled["ARRIVAL_DELAY"]

In [ ]:
X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(X_reg, y_reg, test_size=0.3, random_state=42)

In [ ]:
reg_pipelines = {
    "LinearRegression": Pipeline([
        ("preprocessor", clone(preprocessor_standard)),
        ("regressor", LinearRegression()),
    ]),
    "Ridge": Pipeline([
        ("preprocessor", clone(preprocessor_standard)),
        ("regressor", Ridge()),
    ]),
    "Lasso": Pipeline([
        ("preprocessor", clone(preprocessor_standard)),
        ("regressor", Lasso(max_iter=2000)),
    ]),
}

In [ ]:
reg_param_grids = {
    "LinearRegression": {
        "regressor__fit_intercept": [True, False],
    },
    "Ridge": {
        "regressor__alpha": [0.1, 1.0, 10.0, 100.0],
        "regressor__fit_intercept": [True, False],
    },
    "Lasso": {
        "regressor__alpha": [0.1, 1.0, 10.0, 100.0],
        "regressor__fit_intercept": [True, False],
    },
}

In [ ]:
for model_name, pipeline in reg_pipelines.items():
    print(f"GridSearch - {model_name}")
    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=reg_param_grids[model_name],
        cv=5,
        scoring="neg_mean_squared_error",
        n_jobs=-1
    )
    grid_search.fit(X_reg_train, y_reg_train)

    print("CV Results:")
    cv_split_score_cols = ["split0_test_score", "split1_test_score", "split2_test_score", "split3_test_score", "split4_test_score"]
    df_cv_results = pd.DataFrame(grid_search.cv_results_)[["params"] + cv_split_score_cols + ["mean_test_score"]]
    df_cv_results = pd.concat([
        pd.json_normalize(df_cv_results["params"]),
        np.sqrt(-df_cv_results[cv_split_score_cols + ["mean_test_score"]]),
        np.sqrt(-df_cv_results[cv_split_score_cols]).std(axis=1).rename("std_rmse"),
    ], axis=1)
    df_cv_results.rename(columns=lambda c: c.replace("test_score", "rmse") if "test_score" in c else c, inplace=True)
    print(df_cv_results)
    print("\n")

    print(f"Best Parameters: {grid_search.best_params_}")
    print("\n")

    best_model = grid_search.best_estimator_
    y_reg_pred = best_model.predict(X_reg_test)

    print("Regression Plots:")
    regression_plots(y_reg_test, y_reg_pred, model_name)
    print("\n")

    print("Metrics:")
    print(f"MAE:  {mean_absolute_error(y_reg_test, y_reg_pred):.4f}")
    print(f"RMSE: {np.sqrt(mean_squared_error(y_reg_test, y_reg_pred)):.4f}")
    print("\n")

    print("Feature Importance:")
    df_feature_importance = pd.DataFrame({
        "feature": best_model.named_steps["preprocessor"].get_feature_names_out(),
        "importance": np.abs(best_model.named_steps["regressor"].coef_).flatten(),
    }).sort_values("importance", ascending=False)
    print(df_feature_importance)
    print("\n")